# Assignment 14 — Reflection-Based Two-Agent Conversation (AutoGen)

**Objective:** Simulate a two-agent conversation where a **Content Creator Agent** drafts content on Generative AI and a **Content Critic Agent** evaluates it for *language clarity* and *technical accuracy*, giving constructive feedback. The agents follow the **reflection-based agentic pattern**, and an LLM reflection step summarises the conversation and produces the final refined draft.

**Topic for discussion:** Agentic AI  |  **Framework:** AutoGen (`autogen-agentchat` 0.7.5)  |  **Expected output:** final content shown in Markdown

### How to run
1. **Colab Secrets (recommended):** click the 🔑 icon in the left sidebar, add a secret named `OPENAI_API_KEY`, and turn on *Notebook access*.
2. **Reviewers without the secret:** just run the notebook. If no key is found you will be prompted to paste one (input is hidden and never printed or saved). You can also set `OPENAI_API_KEY` as an environment variable.
3. Run all cells (`Runtime → Run all`). The last cell prints a pass/fail rubric.

### Requirements checklist (verified automatically in the last cell)

| # | Requirement (source) | Where it is satisfied |
|---|---|---|
| R1 | Two agents: Content Creator + Content Critic (Objective) | Agent definitions cell |
| R2 | Clear role for each agent via **system messages** (Directions) | Agent definitions cell |
| R3 | Creator **drafts** initial content on Generative AI (Key Concepts: Drafting) | Conversation cell |
| R4 | Critic **evaluates** for language clarity and technical accuracy (Key Concepts: Evaluation) | Critic system message + conversation |
| R5 | Critic gives **feedback**; Creator **revises** it (Key Concepts: Feedback and Revision) | Conversation cell |
| R6 | **Reflection pattern**: alternating draft → critique → revise loop, ends with a finalized draft | Conversation cell |
| R7 | **Max turns = 3 or 5** (Directions) | `MAX_TURNS` setting + group-chat limit |
| R8 | **Reflection with LLM** as the summary method, capturing feedback, revisions and the final draft (Directions) | Reflection cell |
| R9 | **Topic = Agentic AI** (Topic for discussion) | `TOPIC` setting |
| R10 | Final refined draft **displayed in Markdown** (Expected Output) | Final output cell |


## 1. Install dependencies
Versions are pinned for reproducibility (AutoGen's newer API is not compatible with the legacy `pyautogen` / `autogen` 0.2 packages).

In [ ]:
%pip install -q "autogen-agentchat==0.7.5" "autogen-ext[openai]==0.7.5"

## 2. API key
Looks for the key in **Colab Secrets** first, then an environment variable, then asks you to paste it. The key is never printed.

In [ ]:
import os
from getpass import getpass


def load_api_key() -> str:
    try:
        from google.colab import userdata  # only exists inside Colab
        key = userdata.get("OPENAI_API_KEY")
        if key:
            return key
    except Exception:
        pass  # not in Colab, secret missing, or notebook access not granted
    key = os.environ.get("OPENAI_API_KEY")
    if key:
        return key
    return getpass("OPENAI_API_KEY not found in Colab Secrets or environment. Paste your key: ")


API_KEY = load_api_key().strip()
if not API_KEY:
    raise ValueError("No API key provided. Add OPENAI_API_KEY to Colab Secrets or paste it when prompted.")
print("API key loaded.")

## 3. Settings and model client
All settings live here. `MAX_TURNS` must be **3 or 5** (an odd number so the Creator speaks last and delivers the final draft).

In [ ]:
from autogen_core.models import SystemMessage, UserMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient

MODEL = "gpt-5-mini"   # change here if your account uses a different OpenAI model
MAX_TURNS = 5          # assignment allows 3 or a maximum of 5
TOPIC = "Agentic AI"

if MAX_TURNS not in (3, 5):
    raise ValueError("MAX_TURNS must be 3 or 5 per the assignment directions.")

try:
    model_client = OpenAIChatCompletionClient(model=MODEL, api_key=API_KEY)
except ValueError:
    # AutoGen only knows capabilities for models in its built-in table; describe newer/unlisted ones manually.
    model_client = OpenAIChatCompletionClient(
        model=MODEL,
        api_key=API_KEY,
        model_info={
            "vision": False,
            "function_calling": True,
            "json_output": True,
            "structured_output": True,
            "family": "unknown",
        },
    )

# Fail fast with a clear message if the key or model name is wrong.
try:
    check = await model_client.create([UserMessage(content="Reply with the single word: OK", source="user")])
    print(f"Model '{MODEL}' reachable. Test reply: {str(check.content).strip()[:20]}")
except Exception as exc:
    raise RuntimeError(
        f"Could not call model '{MODEL}'. Check that the API key is valid and the model name in MODEL is available "
        f"to your account. Original error: {type(exc).__name__}: {exc}"
    ) from None

## 4. Define the two agents
Each agent gets a clear role through its **system message**. The Creator's prompt explains the turn structure so it knows when it is drafting, revising, and finalizing; the Critic's prompt makes it assess *language clarity* and *technical accuracy*.

In [ ]:
from autogen_agentchat.agents import AssistantAgent

CREATOR_NAME = "Content_Creator_Agent"
CRITIC_NAME = "Content_Critic_Agent"

creator_turns = ", ".join(str(n) for n in range(1, MAX_TURNS + 1, 2))  # e.g. "1, 3, 5"

creator_system_message = f"""You are the Content Creator Agent. Your role is to draft content on topics involving Generative AI.
Ensure the content is clear, concise, and technically accurate.

You take turns with the Content Critic Agent, and you speak on turns {creator_turns}.
- Turn 1: write the initial draft (a short article of about 150-200 words).
- Middle turns: read the Critic's feedback and revise the draft to address every point.
- Final turn (turn {MAX_TURNS}): produce the final refined draft, incorporating all feedback.

Output rules: respond with ONLY the content, formatted in Markdown (start with a '#' title). Do not add commentary about what you changed."""

critic_system_message = """You are the Content Critic Agent. Your role is to evaluate the content drafted by the Content Creator Agent.
Provide feedback on language clarity and technical correctness, and suggest improvements.

Structure every reply as:
**Language clarity:** your assessment (wording, structure, flow, jargon).
**Technical accuracy:** your assessment (flag anything incorrect, vague, or overstated).
**Suggested improvements:** a short numbered list of concrete changes.

Be constructive and specific. When you evaluate a revised draft, note what improved and what still needs work.
Do not rewrite the whole draft yourself."""

creator_agent = AssistantAgent(name=CREATOR_NAME, model_client=model_client, system_message=creator_system_message)
critic_agent = AssistantAgent(name=CRITIC_NAME, model_client=model_client, system_message=critic_system_message)
print(f"Agents created: {creator_agent.name}, {critic_agent.name}")

## 5. Run the reflection conversation
A `RoundRobinGroupChat` alternates the agents in order: **Creator drafts → Critic evaluates → Creator revises → Critic evaluates → Creator finalizes** (for `MAX_TURNS = 5`). `max_turns` stops the chat after that many agent replies.

In [ ]:
from autogen_agentchat.base import TaskResult
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.teams import RoundRobinGroupChat

TASK = (
    f"Topic: {TOPIC}. "
    "Content Creator Agent: draft a short piece of content about this topic for a Generative AI audience. "
    "Content Critic Agent: evaluate each draft for language clarity and technical accuracy and give feedback. "
    "Creator: revise based on the feedback until the final turn, then deliver the final refined draft."
)

team = RoundRobinGroupChat([creator_agent, critic_agent], max_turns=MAX_TURNS)

transcript = []  # (speaker, text) for every agent turn, in order
print(f"TASK: {TASK}\n")

async for event in team.run_stream(task=TASK):
    if isinstance(event, TaskResult):
        print(f"[Conversation ended: {event.stop_reason}]")
    elif isinstance(event, TextMessage) and event.source in (CREATOR_NAME, CRITIC_NAME):
        transcript.append((event.source, event.content))
        print("=" * 80)
        print(f"TURN {len(transcript)} - {event.source}")
        print("=" * 80)
        print(event.content, "\n")

## 6. Reflection with LLM (summary method)
This reproduces the classic *reflection with LLM* summary: after the conversation, the LLM reflects on the full transcript and writes a summary that captures the **feedback**, the **revisions**, and the **final refined draft**. The final draft itself is taken verbatim from the Creator's last turn so the reflection cannot alter it.

In [ ]:
import re


def strip_code_fence(text: str) -> str:
    """Models sometimes wrap Markdown output in a ```markdown fence; remove it so it renders as Markdown."""
    match = re.match(r"^\s*```(?:markdown|md)?\s*\n(.*?)\n```\s*$", text, flags=re.DOTALL)
    return match.group(1).strip() if match else text.strip()


final_draft = strip_code_fence(transcript[-1][1])  # last turn is the Creator's finalization

transcript_text = "\n\n".join(f"### Turn {i} - {speaker}\n{text}" for i, (speaker, text) in enumerate(transcript, start=1))

reflection_prompt = f"""Below is a conversation in which a Content Creator Agent drafted content on "{TOPIC}" and a Content Critic Agent reviewed it, over {len(transcript)} turns.

Reflect on the conversation and write a concise summary in Markdown with exactly these three sections:
## Feedback Received
Bullet the key feedback the Critic gave (language clarity and technical accuracy).
## Revisions Made
Bullet how the Creator changed the draft in response.
## Reflection
2-3 sentences on how the draft improved overall and any remaining limitations.

Do not reproduce the full draft. Do not invent feedback that is not in the transcript.

--- CONVERSATION ---
{transcript_text}"""

reflection_result = await model_client.create(
    [
        SystemMessage(content="You are a reflective summarizer who accurately summarizes agent conversations."),
        UserMessage(content=reflection_prompt, source="user"),
    ]
)
reflection_summary = str(reflection_result.content).strip()
print("Reflection summary generated.")

## 7. Final output (Markdown)

In [ ]:
from IPython.display import Markdown, display

display(Markdown(f"# Final Refined Draft: {TOPIC}\n\n---\n\n{final_draft}\n\n---\n\n# Conversation Reflection Summary\n\n{reflection_summary}"))

## 8. Rubric verification
Automated check of the requirements table at the top. Every row should print `PASS`.

In [ ]:
speakers = [s for s, _ in transcript]
creator_texts = [t for s, t in transcript if s == CREATOR_NAME]
critic_texts = [t for s, t in transcript if s == CRITIC_NAME]
expected_order = [CREATOR_NAME if i % 2 == 0 else CRITIC_NAME for i in range(MAX_TURNS)]

checks = [
    ("R1  Two agents (Creator + Critic) exist", creator_agent.name == CREATOR_NAME and critic_agent.name == CRITIC_NAME),
    ("R2  Roles defined via system messages",
     "You are the Content Creator Agent" in creator_system_message and "You are the Content Critic Agent" in critic_system_message),
    ("R3  Creator produced an initial draft first", speakers[:1] == [CREATOR_NAME] and len(creator_texts[0].strip()) > 0),
    ("R4  Critic evaluates language clarity AND technical accuracy",
     len(critic_texts) >= 1 and all(len(t.strip()) > 0 for t in critic_texts)
     and "language clarity" in critic_system_message.lower() and "technical" in critic_system_message.lower()),
    ("R5  Creator revised after feedback (2+ Creator turns, draft changed)",
     len(creator_texts) >= 2 and creator_texts[-1].strip() != creator_texts[0].strip()),
    ("R6  Reflection loop alternates Creator/Critic and ends with Creator's final draft",
     speakers == expected_order and speakers[-1] == CREATOR_NAME),
    ("R7  Turns used is 3 or 5 (and matches MAX_TURNS)", len(transcript) in (3, 5) and len(transcript) == MAX_TURNS),
    ("R8  LLM reflection summary captures feedback, revisions, reflection",
     all(h in reflection_summary for h in ("Feedback Received", "Revisions Made", "Reflection"))),
    ("R9  Topic is Agentic AI", TOPIC == "Agentic AI" and "Agentic AI" in TASK),
    ("R10 Final draft is Markdown and was displayed", final_draft.lstrip().startswith("#") and len(final_draft) > 200),
]

for name, ok in checks:
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
print(f"\n{sum(ok for _, ok in checks)}/{len(checks)} checks passed")

In [ ]:
await model_client.close()  # release the HTTP client cleanly